# Solutions — Refs

Only look here after you've actually tried the exercises in `useref.ipynb`.

### LESSON 49 — Exercise

**1. `inputRef.current` on the first render.** It is `null`.

The step that has not happened is the **commit** (LESSON 38). React sets refs during the
commit, so while your component function is running for the very first time there is no DOM
node yet — React has not created it, let alone attached it. The log line reads
`render — inputRef.current is null`, and on every later render it reads `the <input>`, because
by then a commit has been and gone.

**2. Measuring the box.** The number changes when you resize, because it is a fact about the
rendered page rather than about your data. `getBoundingClientRect()` asks the browser what it
actually did after laying everything out.

It could not be kept in state because **nothing in your component knows it**. The width depends
on the window, the font, the parent's padding, the scrollbar — all decided by the browser after
React has handed over the markup. You could only ever put it in state by measuring it first,
which is the thing the ref is for. (And measuring on every render to store it would be
LESSON 44's mistake, with an extra render pass for free.)

**3. Scrolling.**

```jsx
<button id="scroll" onClick={() => boxRef.current.scrollIntoView({ behavior: "smooth" })}>
  scroll to the box
</button>
```

Same three steps, different browser behaviour. Note that `scrollIntoView` is a DOM method, not
a React one — the ref's job is only to hand you the node.

**4. Reading `current` during render.** It throws on the first render:

```text
TypeError: Cannot read properties of null (reading 'value')
```

`current` is `null` until the commit, and the component body runs before it. The smallest fix
is to stop reading it during render — move the line into a handler or an Effect, both of which
run after the commit. (`inputRef.current?.value` would silence the crash, but it would silence
it by always being `undefined` on the first render, which is worse: a bug that no longer
announces itself.)

**5. Renaming `ref` to `inputRef`.** It still works — the focus button still focuses the child.

That is the point of the exercise. In React 19 `ref` is an **ordinary prop**, so the name is
yours to choose; passing `inputRef={someRef}` and accepting `{ inputRef }` behaves identically.
What you lose is the convention every other React developer recognises, and any tooling that
knows what `ref` means. Keep the name `ref` — but now you know it is a habit, not a rule.

Contrast this with `key` (LESSON 20), which really is special: React consumes it and your
component never receives it at all.

**Common mistakes.**

- Reading `ref.current` in the component body and being surprised by `null`.
- Reaching for a ref to read an input's value instead of controlling it. LESSON 30 made the
  input controlled on purpose; a ref would take you back to not knowing what is in it.
- Using a ref to change what is on screen — appending nodes, removing nodes, setting
  `textContent`. React's picture of the page is then wrong and the next render can crash.
- Adding `?.` to make a null ref "safe". It hides the ordering bug rather than fixing it.

### LESSON 49 — Mini challenge

| | tool | why |
|---|---|---|
| 1. focus the first invalid field | **ref** | focus is something you do to a node; it has no JSX representation |
| 2. "copied!" for two seconds | **state** | it is visible UI — it renders, so it is state |
| 3. scroll a chat to the bottom | **ref** | scrolling is a browser behaviour, not a value |
| 4. is a panel open | **state** | it decides what renders |
| 5. how tall a textarea has grown | **ref** to measure, then **state** to share it | the measurement needs the DOM; the shared value needs a render |
| 6. play a `<video>` | **ref** | `play()` is a method on the element; there is no "isPlaying" prop you can set |
| 7. which row is selected | **state** | it changes what the table renders |

**The two of the same shape: 1 and 6** — and 3 belongs with them. Each is *asking or telling
the browser something that has no representation in JSX*. You cannot render "focused" or
"playing"; you call `focus()` and `play()`. That phrase from the lesson is the test, and it is
more reliable than "is it UI?", because focus and playback certainly are UI — they are just not
*rendered* UI.

**The one that tempts people: number 4** — "the panel's open state isn't really data, it's just
a flag." Or number 7, for the same reason.

What goes wrong is that **changing a ref does not re-render** (LESSON 50 gives this its own
lesson). Set `isOpenRef.current = true` and nothing happens on screen: the value is genuinely
updated and the panel stays shut, because nothing told React to call your component again.
Then something *else* triggers a render — an unrelated state change — and the panel silently
pops open, which looks like a haunted app. It is LESSON 25's first lesson all over again: a
value that decides what renders must be state.

### LESSON 50 — Exercise

**1. A plain variable instead of a ref.** The stopwatch starts and then cannot be stopped.

`let intervalId = null` is recreated on every render. The Effect assigns the id to *that
render's* variable; by the time the cleanup runs, the closure it belongs to still holds the id
it was given — but every intervening render made a new variable, and the `setElapsed` calls
have caused plenty of renders. The result is that `clearInterval` is called with whatever that
particular closure captured, which may be `null`, and intervals accumulate.

The console's "stopped" line is misleading because **it prints either way**. The cleanup ran,
the log happened, and `clearInterval` did nothing useful — the message reports that the code
executed, not that the timer stopped. A log line is evidence of a call, not of an effect.

**2. Keeping the id in state.** It works, at a cost: every `setIntervalId` triggers a render
that displays nothing new. You are re-rendering the component to store a number the user will
never see, which is precisely what "not needed for rendering" rules out.

Adding `intervalId` to the dependency array turns it into an infinite loop: the Effect sets the
id, the id changes, the dependency changed, the Effect re-runs, which clears and re-creates the
interval, which sets a new id. The stopwatch either stutters or pegs the CPU.

**3. Fixing the broken counter.** Change `useRef(0)` to `useState(0)` and the click handler to
`setBroken(broken + 1)`. What you needed was the **second** property from the table — the
ability to trigger a render. Retention was never the problem; the ref retained perfectly well.

**4. A render counter.**

```jsx
const renders = useRef(0);
useEffect(() => {
  renders.current = renders.current + 1;
});
```

You cannot increment it during render because the increment would then be part of rendering,
and rendering is supposed to produce the same output for the same inputs. React calls
components more than once — Strict Mode does it deliberately — so a counter incremented during
render counts React's bookkeeping, not the user's experience, and the number would differ
between development and production. That is LESSON 51's rule, arrived at from LESSON 38.

**Common mistakes.**

- Reaching for a ref to avoid a re-render that you actually need. If the screen must change,
  the render is the point, not the cost.
- Storing an id in state and then wondering why the Effect loops.
- Assuming a ref write is "instant" in some way state is not. Both are instant; only one tells
  React.
- Reading `previousCount.current` in the JSX and expecting it to be up to date. It updates in
  an Effect, after the commit — that lag is the feature.

### LESSON 50 — Mini challenge

| | | why |
|---|---|---|
| 1. `setTimeout` id for a toast | **ref** | needed to cancel; never rendered |
| 2. seconds left, shown as a countdown | **state** | it is on screen |
| 3. already shown a one-time tooltip | **ref** | decides behaviour, not output — but see below |
| 4. text in a search box | **state** | controlled input (LESSON 30) |
| 5. the `WebSocket` instance | **ref** | an object you hold to call methods on |
| 6. whether it is connected, shown as a dot | **state** | the dot is on screen |
| 7. scroll position to restore | **ref** | written on scroll, read on return; never rendered |

**Why 5 and 6 differ.** They concern the same connection but not the same *value*. The socket
is a handle you keep in order to call methods on it — the user never learns it exists, so it is
"not needed for rendering". Whether it is connected is drawn on screen as a coloured dot, so it
decides what the user sees, and that makes it state. One connection, one invisible object and
one visible fact.

> Number 3 is worth a second look. A ref is right *if* the tooltip's visibility is driven by
> something else. If the flag itself decides whether the tooltip renders, it is state — the
> same trap as the broken counter.

### LESSON 51 — Exercise

**Part 1 — classify.**

In [ ]:
// 1. formattedTotal ............ PLAIN VARIABLE - computed and used within one render (L29)
// 2. is a modal open ........... STATE - it decides what renders
// 3. the AbortController ....... REF - held to call .abort(); never rendered
// 4. clicks, shown as text ..... STATE - it is on screen
// 5. clicks, sent on unmount ... REF - the number is never displayed, only reported
// 6. a debounce timer id ....... REF - needed to cancel; invisible
// 7. the filtered list ......... NEITHER - derived during render from the data + the filter
//                                (L29/L21). Storing it would be duplicated state.
//
// 4 vs 5: the same count, and the question is never "what is this value?" but "who sees it?".
// Rendered -> state. Reported to analytics on the way out -> a ref, because no render ever
// needs to happen when it changes. Change the requirement to "show a live click count" and
// number 5 becomes state immediately.

console.log("the test is who sees it, not what it is");

**Part 2 — the breakages.**

In [ ]:
// A - writes a ref during render AND renders it.
//     `renderCount.current++` in the body makes the output depend on how many times React
//     called the component, which is not a property of the inputs. Under Strict Mode the
//     number is doubled in development and different in production. Move the increment into
//     an Effect - and accept that the displayed number then lags one render, which is the
//     honest version of this feature.
//
// B - writes a ref during render, and it is the lazy-initialisation pattern people reach for.
//     It looks safe because the write happens only once. It is still a render with a side
//     effect: called twice by Strict Mode, the SECOND call sees a non-null value from the
//     first, which is exactly the cross-call dependency the purity rule forbids. Use
//     `useState(() => Date.now())` if the value is rendered, or set it in an Effect if not.
//
// C - reads AND writes during render, and the bug is visible rather than theoretical.
//     `changed` is computed from a ref that this same render then overwrites, so a second
//     render with the SAME item now reports changed === false. Strict Mode's double call does
//     exactly that, so in development the flash never happens and in production it does -
//     or the reverse. Compare in an Effect and put `changed` in state, or drive the animation
//     from the key (L20) instead.

console.log("handlers, effects and cleanups - never the body");

**Part 3 — the playground.** Keeping the ref and making the screen keep up means adding a
render trigger beside it:

```jsx
const [, forceRender] = useState(0);

<button onClick={() => {
  brokenRef.current = brokenRef.current + 1;
  forceRender((n) => n + 1);
}}>
```

What you have built is `useState` with extra steps, and you would not ship it. Two sources of
truth for one number, a state variable whose value is meaningless, and a reader who has to
understand both to know what the counter does. The exercise is worth doing precisely because
the result is embarrassing — it shows that "use a ref and force an update" is not a clever
optimisation but a reimplementation of the thing you avoided.

### LESSON 51 — Mini challenge

**1. Why state made it slower.** A scroll event fires many times a second — dozens per gesture.
Putting the position in state means a `setState` per event, and therefore a render per event,
each one re-running the component and reconciling the list. The work is real and it is all
wasted, because the scroll position is not drawn anywhere. The list flickers *because* of the
renders, not despite them.

**2. Why a ref restores the wrong position.** It is written on every scroll event and read when
the component mounts or the user returns — two moments that can disagree. Typical causes: the
ref is written during a scroll that is still in progress, so the last value is mid-gesture; or
the component unmounted and remounted, giving a **fresh** ref with its initial value, because a
ref survives renders but not unmounting. "Sometimes" is the tell: the value's lifetime does not
match the thing it is describing.

**3. The actual answer: neither, as posed.** The scroll position is not component state at all
— it belongs to something that outlives this component, so it should be stored where the list's
identity is stored: in a parent that stays mounted, in the router's location state, or in
`sessionStorage`. A ref is right for the *within-a-visit* case, and only once the value is
written at a sensible moment (on unmount or on navigate, not on every event).

What the question teaches is that "state or ref?" is often the wrong question. Both are
component-scoped, and if the value needs to outlive the component, neither is the answer.
Deciding *where a value lives* comes before deciding *which Hook holds it* — which is the
lesson LESSON 29 started and topic 22 finishes.

**4. Reading the ref during render "just to check".** Nothing will warn them, which is the
danger. The component would render from a value React does not track, so React has no reason to
re-render when it changes and no way to know the output is stale. Under Strict Mode the second
call sees whatever the first call left behind, so the rendered result depends on call order —
the exact thing the purity rule forbids. It will appear to work in development, and the failure
in production will be a screen showing a number from a moment that has passed.